In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
# read from the gdg_attendees table 
# in db/applications.db

df = pd.read_sql_query("SELECT * FROM devpost", con="sqlite:///../../../../db/applications.db")

df.head(1)

In [ ]:
tool_counts = (
    df["Which Of The Following Ai Tools Did You Use This Weekend?"]
    .dropna()
    .astype(str)
    .str.replace(r"\band\b", ",", regex=True)
    .str.split(",")
    .explode()
    .str.strip()
)

tool_counts = tool_counts[tool_counts.ne("") & tool_counts.str.lower().ne("n/a")]

tool_counts = (
    tool_counts.value_counts()
    .reset_index()
    .rename(columns={"index": "tool_name", "Which Of The Following Ai Tools Did You Use This Weekend?": "count"})
)

tool_counts

In [ ]:
built_with_counts = (
    df["Built With"]
    .dropna()
    .astype(str)
    .str.replace(r"\band\b", ",", regex=True)
    .str.split(",")
    .explode()
    .str.strip()
)

built_with_counts = built_with_counts[
    built_with_counts.ne("") & built_with_counts.str.lower().ne("n/a")
]

built_with_counts = (
    built_with_counts.value_counts()
    .rename_axis("built_with")
    .reset_index(name="count")
)

# bucket singletons into "Other"
built_with_counts["built_with"] = built_with_counts["built_with"].mask(
    built_with_counts["count"].eq(1), "Other"
)
# bucket doubletons into "Other2"
built_with_counts["built_with"] = built_with_counts["built_with"].mask(
    built_with_counts["count"].eq(2), "Other2"
)

built_with_counts = (
    built_with_counts.groupby("built_with", as_index=False)["count"]
    .sum()
    .sort_values("count", ascending=False)
)

built_with_counts